# End-to-End Gateway Demo

Walking through a complete request lifecycle — from raw query
to final response, showing every decision the gateway makes.

In [1]:
import sys
sys.path.insert(0, '..')

from src.routing.router import ModelRouter
from src.routing.complexity_scorer import ComplexityScorer
from src.cache.semantic_cache import SemanticCache
from src.cost.cost_tracker import CostTracker
from src.gateway.rate_limiter import TokenBucketRateLimiter
import time

router   = ModelRouter()
scorer   = ComplexityScorer()
cache    = SemanticCache()
tracker  = CostTracker()
limiter  = TokenBucketRateLimiter()

cache.clear()
print("All components initialized")

2026-06-23 12:55:37 | INFO | src.routing.router | Initialized ModelRouter
2026-06-23 12:55:37 | INFO | src.cache.semantic_cache | Initialized SemanticCache
2026-06-23 12:55:37 | INFO | src.gateway.rate_limiter | Initialized TokenBucketRateLimiter
2026-06-23 12:55:37 | INFO | src.cache.semantic_cache | Cache cleared
All components initialized


## Step by step: what happens to every request

In [2]:
def simulate_request(query: str, client_id: str = "demo_user", cached_answer: str = None):
    """
    Simulate the full gateway pipeline without calling the LLM.
    Shows every decision made for a given query.
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    # Step 1: Rate limit
    allowed, rate_info = limiter.allow_request(client_id)
    print(f"\n[1] Rate limit: {'ALLOWED' if allowed else 'REJECTED'} "
          f"(remaining tokens: {rate_info['remaining_tokens']})")
    if not allowed:
        print("    → Request rejected, returning 429")
        return

    # Step 2: Complexity scoring
    complexity = scorer.score_detailed(query)
    print(f"\n[2] Complexity score: {complexity.score}")
    print(f"    length={complexity.length_score}, "
          f"keywords={complexity.keyword_score}, "
          f"structure={complexity.structure_score}")
    if complexity.matched_complex_keywords:
        print(f"    Complex keywords: {complexity.matched_complex_keywords}")

    # Step 3: Model routing
    decision = router.route(query)
    print(f"\n[3] Routed to: {decision.model}")
    print(f"    Reason: {decision.reason}")

    # Step 4: Cache lookup
    cached = cache.get(query, model=decision.model)
    if cached:
        answer, similarity = cached
        print(f"\n[4] Cache HIT (similarity={similarity:.4f})")
        print(f"    Returning cached answer immediately")
        tracker.record_request(
            model=decision.model,
            input_tokens=len(query.split()) * 2,
            output_tokens=len(answer.split()) * 2,
            was_cache_hit=True,
        )
        print(f"\n    ANSWER (cached): {answer[:100]}...")
        return

    print(f"\n[4] Cache MISS — would call LLM")

    # Step 5: Simulate LLM call (use provided answer or placeholder)
    answer = cached_answer or f"[Simulated answer for: {query[:40]}...]"
    print(f"\n[5] LLM called → answer received")

    # Step 6: Cache the new answer
    cache.set(query, answer, model=decision.model)
    print(f"\n[6] Answer cached for future requests")

    # Step 7: Record cost
    cost = tracker.record_request(
        model=decision.model,
        input_tokens=len(query.split()) * 4,
        output_tokens=len(answer.split()) * 4,
        was_cache_hit=False,
    )
    print(f"\n[7] Cost recorded: ${cost.total_cost:.6f}")
    print(f"\n    ANSWER: {answer[:100]}...")


# Run through several requests
simulate_request(
    "What is RAG?",
    cached_answer="RAG stands for Retrieval Augmented Generation..."
)

simulate_request(
    "Can you explain what RAG is?",   # should hit cache
)

simulate_request(
    "Design a distributed system for processing 1M events per second with fault tolerance.",
    cached_answer="A distributed event processing system at this scale requires..."
)


QUERY: What is RAG?

[1] Rate limit: ALLOWED (remaining tokens: 9)

[2] Complexity score: 0.1875
    length=0.05, keywords=0.35, structure=0.0
2026-06-23 12:55:52 | INFO | src.routing.router | Routing decision

[3] Routed to: mistralai/Mistral-7B-Instruct-v0.3
    Reason: complexity 0.1875 < threshold 0.5


c:\Users\Snapp\anaconda3\envs\llm-orchestration\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7902.49it/s]



[4] Cache MISS — would call LLM

[5] LLM called → answer received

[6] Answer cached for future requests

[7] Cost recorded: $0.000000

    ANSWER: RAG stands for Retrieval Augmented Generation......

QUERY: Can you explain what RAG is?

[1] Rate limit: ALLOWED (remaining tokens: 9)

[2] Complexity score: 0.275
    length=0.1, keywords=0.5, structure=0.0
2026-06-23 12:56:03 | INFO | src.routing.router | Routing decision

[3] Routed to: mistralai/Mistral-7B-Instruct-v0.3
    Reason: complexity 0.275 < threshold 0.5
2026-06-23 12:56:03 | INFO | src.cache.semantic_cache | Cache HIT

[4] Cache HIT (similarity=0.9456)
    Returning cached answer immediately

    ANSWER (cached): RAG stands for Retrieval Augmented Generation......

QUERY: Design a distributed system for processing 1M events per second with fault tolerance.

[1] Rate limit: ALLOWED (remaining tokens: 8.06)

[2] Complexity score: 0.4292
    length=0.2167, keywords=0.75, structure=0.0
    Complex keywords: ['design']
2026-06-2

In [3]:
tracker.print_report()


COST TRACKING REPORT
  Total requests:        3
  Cache hits:             1
  Cache hit rate:         33.33%
-------------------------------------------------------
  Actual spend:           $0.000000
  Saved by cache:         $0.000000
  Hypothetical (no cache):$0.000000
  Savings:                0.0%
-------------------------------------------------------
  Breakdown by model:
    mistralai/Mistral-7B-Instruct-v0.3: 3 requests, $0.000000 spent


In [4]:
print("\nRate limiting demonstration — 15 rapid requests:\n")

limiter2 = TokenBucketRateLimiter()

for i in range(15):
    allowed, info = limiter2.allow_request("stress_test_user")
    remaining = info['remaining_tokens']
    status    = "✓ ALLOWED" if allowed else "✗ REJECTED"
    print(f"  Request {i+1:>2}: {status} (tokens remaining: {remaining:.2f})")


Rate limiting demonstration — 15 rapid requests:

2026-06-23 12:56:39 | INFO | src.gateway.rate_limiter | Initialized TokenBucketRateLimiter
  Request  1: ✓ ALLOWED (tokens remaining: 9.00)
  Request  2: ✓ ALLOWED (tokens remaining: 8.02)
  Request  3: ✓ ALLOWED (tokens remaining: 7.03)
  Request  4: ✓ ALLOWED (tokens remaining: 6.03)
  Request  5: ✓ ALLOWED (tokens remaining: 5.03)
  Request  6: ✓ ALLOWED (tokens remaining: 4.03)
  Request  7: ✓ ALLOWED (tokens remaining: 3.03)
  Request  8: ✓ ALLOWED (tokens remaining: 2.03)
  Request  9: ✓ ALLOWED (tokens remaining: 1.03)
  Request 10: ✓ ALLOWED (tokens remaining: 0.04)
2026-06-23 12:56:39 | WARNING | src.gateway.rate_limiter | Rate limit exceeded
  Request 11: ✗ REJECTED (tokens remaining: 0.04)
2026-06-23 12:56:39 | WARNING | src.gateway.rate_limiter | Rate limit exceeded
  Request 12: ✗ REJECTED (tokens remaining: 0.04)
2026-06-23 12:56:39 | WARNING | src.gateway.rate_limiter | Rate limit exceeded
  Request 13: ✗ REJECTED (token

## Summary: What the Gateway Does

Every request flows through 7 steps:

| Step | Component | Purpose |
|---|---|---|
| 1 | Rate Limiter | Protect against abuse, control costs |
| 2 | Complexity Scorer | Classify query difficulty |
| 3 | Model Router | Send to cheapest capable model |
| 4 | Semantic Cache | Return cached answer if similar enough |
| 5 | LLM Client | Call the actual model with retry logic |
| 6 | Cache Store | Save answer for future similar queries |
| 7 | Cost Tracker | Record spend and savings analytics |

**The result:** dramatically lower LLM costs with no quality degradation
for the ~65-70% of queries that are either simple or repeat questions.